# Probe construction + real-model evaluation

Thin wrapper — all the logic lives in `src/`. This notebook exists because
Hugging Face is where both HebNLI and the embedding models live, and Colab is
the easiest place with GPU + unrestricted access to it.

**Owner:** Person A (Itay). Sections 1–4 build the probe; 5–6 measure.

There is a **manual step in the middle** (section 3). The miner proposes
candidates; a human decides. That is deliberate — see `src/data/build_probe.py`
for why an automatic filter is not trusted to produce the final probe.

Runtime → Change runtime type → **T4 GPU** before section 5.


## 0. Setup

The repo is **private**, so cloning needs a GitHub token. Create a classic PAT
with `repo` scope at github.com → Settings → Developer settings → Personal access
tokens, and store it as a Colab secret named `GH_TOKEN` (key icon, left sidebar).

The token is read from the secret and never written into a cell, so it does not
end up in the saved notebook or in git.


In [ ]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'person-a'

import os, subprocess
from google.colab import userdata

gh_token = userdata.get('GH_TOKEN').strip()
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(f'/content/{REPO}')
subprocess.run(['git','pull','-q','origin',BRANCH], check=True)

# drop the token from the stored remote so it is not left on disk
subprocess.run(['git','remote','set-url','origin',
                f'https://github.com/{OWNER}/{REPO}.git'], check=True)

!pip install -q -r requirements.txt datasets
!git log --oneline -3


In [ ]:
# offline sanity checks first — if these fail, stop and fix before burning GPU time
!python -m src.data.negation --selftest
!python -m tests.test_data_pipeline | tail -3
!python -m tests.test_projection | tail -3


## 1. Hugging Face access

The HebNLI dataset card is marked private, so `load_dataset` needs a token.
Create one at huggingface.co → Settings → Access Tokens (read scope is enough)
and store it as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).


In [ ]:
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()

from huggingface_hub import whoami
print(whoami(token=os.environ['HF_TOKEN'])['name'])


## 2. Mine candidates from HebNLI

Prints a funnel showing how many pairs die at each filter. Most die at
`negation_added` — that is the point: HebNLI labels antonyms, number swaps and
entity swaps as `contradiction`, and none of those belong in a negation probe.

The funnel numbers go in the dataset section of the report, so keep the output.


In [ ]:
!python -m src.data.hebnli --split train --out data/raw/hebnli_train.jsonl


In [ ]:
# mining from the cached copy — no second download
!python -m src.data.build_probe mine \
    --source data/raw/hebnli_train.jsonl \
    --out data/probe/review.csv \
    --stats-out results/probe_funnel.json


In [ ]:
# eyeball a few before committing to a review pass
import pandas as pd
df = pd.read_csv('data/probe/review.csv')
print(len(df), 'candidates;', (df.paraphrase.isna()).sum(), 'need a paraphrase')
df.head(15)[['id','target','paraphrase','negation','neg_markers','containment']]


## 3. Review — the manual step

Download `review.csv`, open it in Sheets or Excel (it is UTF-8 with BOM, so
Hebrew opens correctly), and for each row:

- set **`keep`** to `y` or `n`
- if `note` says `WRITE PARAPHRASE`, write one: same meaning, different wording
- fix any negation that changes more than the polarity — CONDAQA-style, *only*
  the negation may differ from the target

Reject anything where the opposition comes from an antonym, a number, or world
knowledge rather than the negation marker itself.

Target for M1: **~100 clean triples**. Have Shachar double-annotate a sample and
record the agreement rate — that number goes in the report.

Then re-upload the edited file.


In [ ]:
from google.colab import files
files.download('data/probe/review.csv')


In [ ]:
# after editing, upload the reviewed file back
from google.colab import files
uploaded = files.upload()          # pick your edited review.csv
name = next(iter(uploaded))
!cp "$name" data/probe/review_done.csv
print('uploaded', name)


## 4. Finalize → probe.jsonl + splits

The split is deterministic and stratified by negation type, so a rerun cannot
silently reshuffle what the projection was fitted on, and no negation type ends
up entirely on one side.


In [ ]:
!python -m src.data.build_probe finalize \
    --review data/probe/review_done.csv \
    --out data/probe/probe.jsonl
!python -m src.data.build_probe validate --probe data/probe/probe.jsonl


## 5. Baseline measurement on real models

**This is the M1 sync point** — B's harness meeting A's data for the first time.
First download will be a few GB; on a fresh runtime expect a wait.

The headline number: how small is the paraphrase-vs-negation gap. Email it to David.


In [ ]:
!python -m src.harness.run_eval \
    --probe data/probe/probe.jsonl \
    --models multilingual-e5 labse \
    --interventions baseline \
    --out results/results_baseline.csv

import pandas as pd
pd.read_csv('results/results_baseline.csv')


## 6. Projection ablation

Direction × centring × γ selection, all measured on the **test** split.

Two things to look at before believing any row:
- a `*` means γ landed on the top of the sweep grid — the grid picked it, not
  the data. Widen `DEFAULT_SCALE_GRID` and re-run.
- compare `/cv` against `/train`. The gap between them is how much of the naive
  result was the selection fitting itself.


In [ ]:
!python -m src.interventions.projection_report \
    --probe data/probe/probe.jsonl \
    --models multilingual-e5 labse \
    --out results/projection_ablation.csv \
    --show-sweeps


## 7. Keep the results

`results/*.csv` is small and belongs in git so both of us see the same numbers.
Model weights and `data/raw/` do not — `.gitignore` already blocks them.


In [ ]:
from google.colab import files
files.download('results/projection_ablation.csv')
files.download('results/results_baseline.csv')
files.download('results/probe_funnel.json')
files.download('data/probe/probe.jsonl')


Commit them from your machine on `person-a`, then push. Prefix the message
`data:` or `projection:` per CONTRIBUTING.md.
